# Next Word Predictor using LSTM

# Import Libraries

In [117]:
import pandas as pd
import numpy as np

In [118]:
from sklearn.model_selection import train_test_split

In [119]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [120]:
import re
import time

# Check Device Availability

In [121]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

# Make Data

In [122]:
with open("data.txt", "r", encoding="utf-8") as f:
    text = f.read()
len(text)

54521

# Split Sentences

In [123]:
def split_sentences(text: str):
    # Custom rule: split on . ! ? or newlines
    return re.split(r'(?<=[.!?])\s+|\n+', text)

# Data Preprocessing

In [124]:
def split_sentences(text: str):
    # Custom rule: split on . ! ? or newlines
    return re.split(r'(?<=[.!?])\s+|\n+', text)

def preprocess_text(text: str) -> str:

    text = text.lower()

    text = re.sub(r'[\\|]', '', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)

    text = re.sub(r"can't", "can not", text)
    text = re.sub(r"won't", "will not", text)

    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\d+', '', text)

    text = re.sub(r'[ \t]+', ' ', text)  # keep newline
    text = text.strip()

    return text

def preprocess_document(text: str) -> list:
    sentences = split_sentences(text)
    return [preprocess_text(s) for s in sentences]

In [125]:
preprocessed_sentences = preprocess_document(text)
len(preprocessed_sentences)

552

# Tokenizer

In [126]:
def tokenize(text: list) -> list[str]:
  tokens = []
  for sentence in text:
    token = sentence.split()
    tokens.extend(token)
  return tokens

In [127]:
tokens = tokenize(preprocessed_sentences)
len(tokens)

8148

# Vocabulary

In [128]:
def make_vocabulary(tokens: list) -> dict:
  vocabulary = {"<unk>": 0}

  for token in tokens:
    if token not in vocabulary:
      vocabulary[token] = len(vocabulary)
  return vocabulary

In [129]:
vocabulary = make_vocabulary(tokens)
len(vocabulary)

1912

# String Sentence to Number Sentence

In [130]:
def text_to_indices(preprocessed_sentences: list[str], vocabulary: dict[str, int]) -> list[list[int]]:
    """
    Convert each preprocessed sentence into a list of integer token indices
    using the provided vocabulary. Unknown tokens map to 0.

    Args:
        preprocessed_sentences (list[str]): List of pre-tokenized, preprocessed sentences.
        vocabulary (dict[str, int]): Mapping from token to numeric index.

    Returns:
        list[list[int]]: A list of numeric sequences.
    """

    input_numeric_sentences = []

    for sentence in preprocessed_sentences:
        # Handle empty or whitespace-only sentences
        if sentence.strip() == "":
            input_numeric_sentences.append([0])
            continue

        sequence = [
            vocabulary.get(token, 0)   # Use 0 for unknown tokens
            for token in sentence.split()
        ]

        input_numeric_sentences.append(sequence)

    return input_numeric_sentences

In [131]:
x = ["hello visit have nlp fun", " ", "nlp is also an acronym for "]
text_to_indices(x, vocabulary)

[[0, 0, 57, 4, 0], [0], [4, 5, 292, 107, 0, 157]]

In [132]:
preprocessed_sentences[17]

'current systems are prone to bias and incoherence and occasionally behave erratically'

In [133]:
input_numaric_sentences = text_to_indices(preprocessed_sentences, vocabulary)
input_numaric_sentences[17]

[215, 216, 79, 217, 32, 218, 19, 219, 19, 220, 221, 222]

# Make Training Sequence (Target Columns)

In [134]:
def training_sequence(numaric_sentence: list[list[int]]) -> list[list[int]]:
  training_sequences = []
  for sentence in numaric_sentence:
    for i in range(1, len(sentence)):
      # print(i[j], end=" ")
      training_sequences.append(sentence[ : i+1])
  return training_sequences

In [135]:
training_sequences = training_sequence(input_numaric_sentences)
training_sequences[17]

[1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 2, 14, 6, 15]

In [136]:
len(training_sequences)

7614

# Padding

In [137]:
max_len = max([len(s) for s in training_sequences])
max_len

51

In [138]:
[0]* (max_len - len(training_sequences[7])) + training_sequences[7]
#

[0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 2,
 3,
 5,
 6,
 7]

In [139]:
def padded_sequences(training_sequences, max_len):
  # max_len = max([len(s) for s in training_sequences])
  seq = []
  for sequence in training_sequences:
    with_pad = [0]* (max_len - len(sequence)) + sequence
    seq.append(with_pad)
  return torch.tensor(seq)

In [140]:
training_padded_sequences = padded_sequences(training_sequences, max_len)
training_padded_sequences

tensor([[   0,    0,    0,  ...,    0,    1,    2],
        [   0,    0,    0,  ...,    1,    2,    3],
        [   0,    0,    0,  ...,    2,    3,    4],
        ...,
        [   0,    0,    0,  ..., 1433,   79, 1910],
        [   0,    0,    0,  ...,   79, 1910,  157],
        [   0,    0,    0,  ..., 1910,  157, 1911]])

In [141]:
len(training_padded_sequences[40])

51

# Separate `X`, `y`

In [142]:
X = training_padded_sequences[:, : -1]
y = training_padded_sequences[:, -1]

# Train Test Split

In [143]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=17)
X_train.shape, X_test.shape

(torch.Size([6091, 50]), torch.Size([1523, 50]))

# Custom Dataset Class

In [144]:
class WordPredDataset(Dataset):
  def __init__(self, features, labels):
    self.features = features
    self.labels = labels

  def __len__(self):
    return len(self.features)

  def __getitem__(self, index):
    return self.features[index], self.labels[index]

In [145]:
train_dataset = WordPredDataset(X_train, y_train)
test_dataset = WordPredDataset(X_test, y_test)

# DataLoader Class

In [146]:
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=16
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=16
)

# LSTM Model Build

In [147]:
# class MyLSTM(nn.Module):
#   def __init__(self, vocab_size) -> None:
#      super(MyLSTM, self).__init__()

#      self.embedding = nn.Embedding(vocab_size, 100)
#      self.lstm1 = nn.LSTM(100, 128, batch_first=True)
#      self.lstm2 = nn.LSTM(128, 256, batch_first=True)
#      self.output = nn.Linear(256, vocab_size)

#   def forward(self, features):
#     embedded = self.embedding(features)
#     inter_hidden_state, (final_hidden_state, final_cell_state) = self.lstm(embedded)
#     output = self.output(final_hidden_state.squeeze(0))
#     return output

In [148]:
class MyLSTM(nn.Module):
    def __init__(self, vocab_size):
        super(MyLSTM, self).__init__()

        embed_dim = 100
        hidden1 = 128
        hidden2 = 256

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.lstm1 = nn.LSTM(embed_dim, hidden1, batch_first=True)
        self.lstm2 = nn.LSTM(hidden1, hidden2, batch_first=True)

        # final prediction
        self.output = nn.Linear(hidden2, vocab_size)

    def forward(self, features):
        # features shape: (batch, seq_len)

        embedded = self.embedding(features)  # (batch, seq_len, embed_dim)

        out1, (h1, c1) = self.lstm1(embedded)   # h1: (1, batch, 128)
        out2, (h2, c2) = self.lstm2(out1)       # h2: (1, batch, 256)

        # Use final hidden state of the last LSTM layer
        final_hidden = h2[-1]  # shape: (batch, 256)

        logits = self.output(final_hidden)  # (batch, vocab_size)
        return logits

## Model Creation

In [149]:
model = MyLSTM(len(vocabulary)).to(device)
model

MyLSTM(
  (embedding): Embedding(1912, 100)
  (lstm1): LSTM(100, 128, batch_first=True)
  (lstm2): LSTM(128, 256, batch_first=True)
  (output): Linear(in_features=256, out_features=1912, bias=True)
)

## Parameters Init

In [150]:
epochs = 100
learning_rate = 0.01

## Loss and Optimizer

In [151]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), learning_rate)

## Training Loop

In [152]:
for epoch in range(epochs):

    model.train()
    losses = []

    for features, labels in train_loader:

        # Move to GPU/CPU
        features = features.to(device)
        labels   = labels.to(device)

        # Forward pass
        y_pred = model(features)

        # Compute loss
        loss = criterion(y_pred, labels)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()

        # Update weights
        optimizer.step()

        losses.append(loss.item())

    avg_loss = np.mean(losses)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

Epoch 1/100, Loss: 6.5589
Epoch 2/100, Loss: 5.9449
Epoch 3/100, Loss: 5.6705
Epoch 4/100, Loss: 5.2635
Epoch 5/100, Loss: 4.7713
Epoch 6/100, Loss: 4.4000
Epoch 7/100, Loss: 4.1095
Epoch 8/100, Loss: 3.6082
Epoch 9/100, Loss: 3.2679
Epoch 10/100, Loss: 2.9380
Epoch 11/100, Loss: 2.7025
Epoch 12/100, Loss: 2.2843
Epoch 13/100, Loss: 2.1452
Epoch 14/100, Loss: 1.9558
Epoch 15/100, Loss: 1.7573
Epoch 16/100, Loss: 1.7275
Epoch 17/100, Loss: 1.6736
Epoch 18/100, Loss: 1.7461
Epoch 19/100, Loss: 1.5661
Epoch 20/100, Loss: 1.2797
Epoch 21/100, Loss: 1.1168
Epoch 22/100, Loss: 1.0508
Epoch 23/100, Loss: 0.9629
Epoch 24/100, Loss: 1.0776
Epoch 25/100, Loss: 1.2238
Epoch 26/100, Loss: 1.2796
Epoch 27/100, Loss: 1.3367
Epoch 28/100, Loss: 1.3591
Epoch 29/100, Loss: 1.2060
Epoch 30/100, Loss: 1.0755
Epoch 31/100, Loss: 0.9292
Epoch 32/100, Loss: 0.9405
Epoch 33/100, Loss: 0.8339
Epoch 34/100, Loss: 0.8041
Epoch 35/100, Loss: 1.1056
Epoch 36/100, Loss: 1.3069
Epoch 37/100, Loss: 1.2884
Epoch 38/1

# Evaluation

In [153]:
correct_predictions_count = 0
total_samples = 0

model.eval()  # Set model to evaluation mode

with torch.no_grad():  # Disable gradient calculation
    for features, labels in test_loader:

        features = features.to(device)
        labels = labels.to(device)

        outputs = model(features)

        # Get predicted classes
        _, predicted = torch.max(outputs, dim=1)

        # Update counters
        correct_predictions_count += (predicted == labels).sum().item()
        total_samples += labels.size(0)

# Compute accuracy
accuracy = correct_predictions_count / total_samples if total_samples > 0 else 0
print(f"Overall Accuracy: {accuracy:.4f}")

Overall Accuracy: 0.0735


# Predictions

In [154]:
def prediction(model: nn.Module, vocab: dict, text: str, max_len: int, device='cpu'):
    model.eval()

    # Inverse vocabulary (index → token)
    idx2word = {idx: word for word, idx in vocab.items()}

    # Preprocess
    prep_text = preprocess_document(text)

    # Convert to indices
    numeric_seq = text_to_indices(prep_text, vocab)      # input must be list[str]

    # Pad
    padded = padded_sequences(numeric_seq, max_len)         # returns numpy or list?

    # Convert to tensor
    padded = torch.tensor(padded, dtype=torch.long).to(device)

    # Model prediction
    with torch.no_grad():
        output = model(padded)

    # Get predicted class index
    _, pred_idx = torch.max(output, dim=1)
    pred_idx = pred_idx.item()

    # Convert index → label/token
    pred_word = idx2word.get(pred_idx, "<UNK>")

    return f"{text} {pred_word}"

In [156]:
total_word = 15
pred_text = "Word2Vec, introduced in 2013,"

for i in range(total_word):
  next_word = prediction(model, vocabulary, pred_text, max_len, device)
  print(next_word)
  pred_text = next_word
  time.sleep(1)

/tmp/ipython-input-686225401.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  padded = torch.tensor(padded, dtype=torch.long).to(device)


Word2Vec, introduced in 2013, uses
Word2Vec, introduced in 2013, uses deep
Word2Vec, introduced in 2013, uses deep can
Word2Vec, introduced in 2013, uses deep can be
Word2Vec, introduced in 2013, uses deep can be applied
Word2Vec, introduced in 2013, uses deep can be applied to
Word2Vec, introduced in 2013, uses deep can be applied to solve
Word2Vec, introduced in 2013, uses deep can be applied to solve problems
Word2Vec, introduced in 2013, uses deep can be applied to solve problems such
Word2Vec, introduced in 2013, uses deep can be applied to solve problems such as
Word2Vec, introduced in 2013, uses deep can be applied to solve problems such as sentiment
Word2Vec, introduced in 2013, uses deep can be applied to solve problems such as sentiment and
Word2Vec, introduced in 2013, uses deep can be applied to solve problems such as sentiment and employee
Word2Vec, introduced in 2013, uses deep can be applied to solve problems such as sentiment and employee efforts
Word2Vec, introduced in